In [1]:
from f5_tts.model.cfm import CFM

from f5_tts.model.backbones.unett import UNetT
from f5_tts.model.backbones.dit import DiT
from f5_tts.model.backbones.mmdit import MMDiT

from f5_tts.model.trainer import Trainer


import os
import sys

# sys.path.append(f"../../{os.path.dirname(os.path.abspath(__file__))}/third_party/BigVGAN/")

import hashlib
import re
import tempfile
from importlib.resources import files

import matplotlib

matplotlib.use("Agg")

import matplotlib.pylab as plt
import numpy as np
import torch
import torchaudio
import tqdm
from pydub import AudioSegment, silence
from transformers import pipeline
from vocos import Vocos

# from f5_tts.model import CFM
from num2words import num2words
import soundfile as sf
# import gradio as gr

2025-05-21 02:22:14.164637: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1747794134.211331     357 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1747794134.224882     357 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1747794134.324958     357 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1747794134.324979     357 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1747794134.324981     357 computation_placer.cc:177] computation placer alr

In [2]:
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(device)
# -----------------------------------------

target_sample_rate = 24000
n_mel_channels = 100
hop_length = 256
win_length = 1024
n_fft = 1024
mel_spec_type = "vocos"
# mel_spec_type = "bigvgan"
target_rms = 0.1
cross_fade_duration = 0.15
ode_method = "euler"
nfe_step = 32  # 16, 32
cfg_strength = 2.0
sway_sampling_coef = -1.0
speed = 1.0
fix_duration = None

# -----------------------------------------

_ref_audio_cache = {}
# load asr pipeline
asr_pipe = None

cuda


In [24]:
#UTILS_INFER
# chunk text into smaller pieces

def chunk_text(text, max_chars=135):
    """
    Splits the input text into chunks, each with a maximum number of characters.

    Args:
        text (str): The text to be split.
        max_chars (int): The maximum number of characters per chunk.

    Returns:
        List[str]: A list of text chunks.
    """
    chunks = []
    current_chunk = ""
    # Split the text into sentences based on punctuation followed by whitespace
    # sentences = re.split(r"(?<=[;:,.!?])\s+|(?<=[；：，。！？])", text)
    sentences = re.split(r"(?<=[;:.!?])\s+|(?<=[；：。！？])", text)
    for sentence in sentences:
        if len(current_chunk.encode("utf-8")) + len(sentence.encode("utf-8")) <= max_chars:
            current_chunk += sentence + " " if sentence and len(sentence[-1].encode("utf-8")) == 1 else sentence
        else:
            if current_chunk:
                chunks.append(current_chunk.strip())
            current_chunk = sentence + " " if sentence and len(sentence[-1].encode("utf-8")) == 1 else sentence

    if current_chunk:
        chunks.append(current_chunk.strip())

    return chunks


def sentences_text(text,max_chars=135):
    """
    Splits the input text into chunks, each with a maximum number of characters.

    Args:
        text (str): The text to be split.
        max_chars (int): The maximum number of characters per chunk.

    Returns:
        List[str]: A list of text chunks.
    """
    
    # Split the text into sentences based on punctuation followed by whitespace
    # sentences = re.split(r"(?<=[;:,.!?])\s+|(?<=[；：，。！？])", text)
    sentences = re.split(r"(?<=[;:.!?])\s+|(?<=[；：。！？])", text)

    return sentences



# load vocoder
def load_vocoder(is_local=False, local_path="", device=device):
    if mel_spec_type == "vocos":
        if is_local:
            print(f"Load vocos from local path {local_path}")
            vocoder = Vocos.from_hparams(f"{local_path}/config.yaml")
            state_dict = torch.load(f"{local_path}/pytorch_model.bin", map_location="cpu")
            vocoder.load_state_dict(state_dict)
            vocoder = vocoder.eval().to(device)
        else:
            print("Download Vocos from huggingface charactr/vocos-mel-24khz")
            vocoder = Vocos.from_pretrained("charactr/vocos-mel-24khz").to(device)
    elif mel_spec_type == "bigvgan":
        try:
            # from third_party.BigVGAN import bigvgan
            import bigvgan
        except ImportError:
            print("You need to follow the README to init submodule and change the BigVGAN source code.")
        if is_local:
            """download from https://huggingface.co/nvidia/bigvgan_v2_24khz_100band_256x/tree/main"""
            vocoder = bigvgan.BigVGAN.from_pretrained(local_path, use_cuda_kernel=False)
        else:
            vocoder = bigvgan.BigVGAN.from_pretrained("nvidia/bigvgan_v2_24khz_100band_256x", use_cuda_kernel=False)

        vocoder.remove_weight_norm()
        vocoder = vocoder.eval().to(device)
    return vocoder



def initialize_asr_pipeline(device=device, dtype=None):
    if dtype is None:
        dtype = (
            torch.float16 if device == "cuda" and torch.cuda.get_device_properties(device).major >= 6 else torch.float32
        )
    global asr_pipe
    asr_pipe = pipeline(
        "automatic-speech-recognition",
        model="openai/whisper-large-v3-turbo",
        torch_dtype=dtype,
        device=device,
    )


# load model checkpoint for inference


def load_checkpoint(model, ckpt_path, device, dtype=None, use_ema=True):
    if dtype is None:
        dtype = (
            torch.float16 if device == "cuda" and torch.cuda.get_device_properties(device).major >= 6 else torch.float32
        )
    model = model.to(dtype)

    ckpt_type = ckpt_path.split(".")[-1]
    if ckpt_type == "safetensors":
        from safetensors.torch import load_file

        checkpoint = load_file(ckpt_path)
    else:
        checkpoint = torch.load(ckpt_path, weights_only=True)

    if use_ema:
        if ckpt_type == "safetensors":
            checkpoint = {"ema_model_state_dict": checkpoint}
        checkpoint["model_state_dict"] = {
            k.replace("ema_model.", ""): v
            for k, v in checkpoint["ema_model_state_dict"].items()
            if k not in ["initted", "step"]
        }

        # patch for backward compatibility, 305e3ea
        for key in ["mel_spec.mel_stft.mel_scale.fb", "mel_spec.mel_stft.spectrogram.window"]:
            if key in checkpoint["model_state_dict"]:
                del checkpoint["model_state_dict"][key]

        model.load_state_dict(checkpoint["model_state_dict"])
    else:
        if ckpt_type == "safetensors":
            checkpoint = {"model_state_dict": checkpoint}
        model.load_state_dict(checkpoint["model_state_dict"])

    return model.to(device)


# load model for inference



def remove_silence_edges(audio, silence_threshold=-42):
    # Remove silence from the start
    non_silent_start_idx = silence.detect_leading_silence(audio, silence_threshold=silence_threshold)
    audio = audio[non_silent_start_idx:]

    # Remove silence from the end
    non_silent_end_duration = audio.duration_seconds
    for ms in reversed(audio):
        if ms.dBFS > silence_threshold:
            break
        non_silent_end_duration -= 0.001
    trimmed_audio = audio[: int(non_silent_end_duration * 1000)]

    return trimmed_audio



# infer process: chunk text -> infer batches [i.e. infer_batch_process()]

# remove silence from generated wav


def remove_silence_for_generated_wav(filename):
    aseg = AudioSegment.from_file(filename)
    non_silent_segs = silence.split_on_silence(
        aseg, min_silence_len=1000, silence_thresh=-50, keep_silence=500, seek_step=10
    )
    non_silent_wave = AudioSegment.silent(duration=0)
    for non_silent_seg in non_silent_segs:
        non_silent_wave += non_silent_seg
    aseg = non_silent_wave
    aseg.export(filename, format="wav")


# save spectrogram


def save_spectrogram(spectrogram, path):
    plt.figure(figsize=(12, 4))
    plt.imshow(spectrogram, origin="lower", aspect="auto")
    plt.colorbar()
    plt.savefig(path)
    plt.close()



In [4]:
#UTILS

import os
import random
from collections import defaultdict
from importlib.resources import files

import torch
from torch.nn.utils.rnn import pad_sequence

import jieba
from pypinyin import lazy_pinyin, Style


# seed everything
def seed_everything(seed=0):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# helpers


def exists(v):
    return v is not None


def default(v, d):
    return v if exists(v) else d


def traducir_numero_a_texto(texto):
    texto_separado = re.sub(r'([A-Za-z])(\d)', r'\1 \2', texto)
    texto_separado = re.sub(r'(\d)([A-Za-z])', r'\1 \2', texto_separado)
    
    def reemplazar_numero(match):
        numero = match.group()
        return num2words(int(numero), lang='es')

    texto_traducido = re.sub(r'\b\d+\b', reemplazar_numero, texto_separado)

    return texto_traducido


# convert char to pinyin
def convert_char_to_pinyin(text_list, polyphone=True):
    final_text_list = []
    god_knows_why_en_testset_contains_zh_quote = str.maketrans(
        {"“": '"', "”": '"', "‘": "'", "’": "'"}
    )  # in case librispeech (orig no-pc) test-clean
    custom_trans = str.maketrans({";": ","})  # add custom trans here, to address oov
    for text in text_list:
        char_list = []
        text = text.translate(god_knows_why_en_testset_contains_zh_quote)
        text = text.translate(custom_trans)
        for seg in jieba.cut(text):
            seg_byte_len = len(bytes(seg, "UTF-8"))
            if seg_byte_len == len(seg):  # if pure alphabets and symbols
                if char_list and seg_byte_len > 1 and char_list[-1] not in " :'\"":
                    char_list.append(" ")
                char_list.extend(seg)
            elif polyphone and seg_byte_len == 3 * len(seg):  # if pure chinese characters
                seg = lazy_pinyin(seg, style=Style.TONE3, tone_sandhi=True)
                for c in seg:
                    if c not in "。，、；：？！《》【】—...":
                        char_list.append(" ")
                    char_list.append(c)
            else:  # if mixed chinese characters, alphabets and symbols
                for c in seg:
                    if ord(c) < 256:
                        char_list.extend(c)
                    else:
                        if c not in "。，、；：？！《》【】—...":
                            char_list.append(" ")
                            char_list.extend(lazy_pinyin(c, style=Style.TONE3, tone_sandhi=True))
                        else:  # if is zh punc
                            char_list.append(c)
        final_text_list.append(char_list)

    return final_text_list


# filter func for dirty data with many repetitions


def repetition_found(text, length=2, tolerance=10):
    pattern_count = defaultdict(int)
    for i in range(len(text) - length + 1):
        pattern = text[i : i + length]
        pattern_count[pattern] += 1
    for pattern, count in pattern_count.items():
        if count > tolerance:
            return True
    return False



In [5]:
def get_tokenizer(dataset_name, tokenizer: str = "pinyin"):
    """
    tokenizer   - "pinyin" do g2p for only chinese characters, need .txt vocab_file
                - "char" for char-wise tokenizer, need .txt vocab_file
                - "byte" for utf-8 tokenizer
                - "custom" if you're directly passing in a path to the vocab.txt you want to use
    vocab_size  - if use "pinyin", all available pinyin types, common alphabets (also those with accent) and symbols
                - if use "char", derived from unfiltered character & symbol counts of custom dataset
                - if use "byte", set to 256 (unicode byte range)
    """
    if tokenizer in ["pinyin", "char"]:
        tokenizer_path = os.path.join(files("f5_tts").joinpath("../../data"), f"{dataset_name}_{tokenizer}/vocab.txt")
        # tokenizer_path ="./F5TTS/vocab_pinyin.txt"
        # tokenizer_path ="/home/jupyter/F5TTS/vocab_pinyin.txt"
        with open(tokenizer_path, "r", encoding="utf-8") as f:
            vocab_char_map = {}
            for i, char in enumerate(f):
                vocab_char_map[char[:-1]] = i
        vocab_size = len(vocab_char_map)
        assert vocab_char_map[" "] == 0, "make sure space is of idx 0 in vocab.txt, cuz 0 is used for unknown char"

    elif tokenizer == "byte":
        vocab_char_map = None
        vocab_size = 256

    elif tokenizer == "custom":
        with open(dataset_name, "r", encoding="utf-8") as f:
            vocab_char_map = {}
            for i, char in enumerate(f):
                vocab_char_map[char[:-1]] = i
        vocab_size = len(vocab_char_map)

    return vocab_char_map, vocab_size


In [6]:
def preprocess_ref_audio_text(ref_audio_orig, ref_text, clip_short=False, show_info=print, device=device):
    show_info("Converting audio...")
    print("Converting audio...")
    with tempfile.NamedTemporaryFile(delete=False, suffix=".wav") as f:
        aseg = AudioSegment.from_file(ref_audio_orig)

        if clip_short:
            # 1. try to find long silence for clipping
            non_silent_segs = silence.split_on_silence(
                aseg, min_silence_len=1000, silence_thresh=-50, keep_silence=1000, seek_step=10 
                # aseg, min_silence_len=2000, silence_thresh=-50, keep_silence=1000, seek_step=10 
            )
            non_silent_wave = AudioSegment.silent(duration=0)
            for non_silent_seg in non_silent_segs:
                if len(non_silent_wave) > 6000 and len(non_silent_wave + non_silent_seg) > 15000:
                    show_info("Audio is over 15s, clipping short. (1)")
                    break
                non_silent_wave += non_silent_seg

            # 2. try to find short silence for clipping if 1. failed
            if len(non_silent_wave) > 15000:
                non_silent_segs = silence.split_on_silence(
                    aseg, min_silence_len=100, silence_thresh=-40, keep_silence=1000, seek_step=10
                )
                non_silent_wave = AudioSegment.silent(duration=0)
                for non_silent_seg in non_silent_segs:
                    if len(non_silent_wave) > 6000 and len(non_silent_wave + non_silent_seg) > 15000:
                        show_info("Audio is over 15s, clipping short. (2)")
                        break
                    non_silent_wave += non_silent_seg

            aseg = non_silent_wave

            # 3. if no proper silence found for clipping
            if len(aseg) > 15000:
                aseg = aseg[:15000]
                show_info("Audio is over 15s, clipping short. (3)")

        aseg = remove_silence_edges(aseg) + AudioSegment.silent(duration=50)
        aseg.export(f.name, format="wav")
        ref_audio = f.name

    # Compute a hash of the reference audio file
    with open(ref_audio, "rb") as audio_file:
        audio_data = audio_file.read()
        audio_hash = hashlib.md5(audio_data).hexdigest()

    global _ref_audio_cache
    if audio_hash in _ref_audio_cache:
        # Use cached reference text
        show_info("Using cached reference text...")
        ref_text = _ref_audio_cache[audio_hash]
    else:
        print("ref_text ",len(ref_text),ref_text)
        if not ref_text.strip():
        # if len(ref_text)==0:
        # if ref_text =="":
            
            global asr_pipe
            if asr_pipe is None:
                initialize_asr_pipeline(device=device)
            show_info("No reference text provided, transcribing reference audio...")
            ref_text = asr_pipe(
                ref_audio,
                chunk_length_s=30,
                batch_size=128,
                generate_kwargs={"task": "transcribe"},
                return_timestamps=False,
            )["text"].strip()
            
            show_info("Finished transcription")
        else:
            show_info("Using custom reference text...")
        # Cache the transcribed text
        _ref_audio_cache[audio_hash] = ref_text
    print("ref_text: ",ref_text)
    # Ensure ref_text ends with a proper sentence-ending punctuation
    if not ref_text.endswith(". ") and not ref_text.endswith("。"):
        if ref_text.endswith("."):
            ref_text += " "
        else:
            ref_text += ". "

    return ref_audio, ref_text


In [7]:
def load_model(
    model_cls,
    model_cfg,
    ckpt_path,
    mel_spec_type=mel_spec_type,
    vocab_file="",
    ode_method=ode_method,
    use_ema=True,
    device=device,
):
    if vocab_file == "":
        # vocab_file = str(files("f5_tts").joinpath("infer/examples/vocab.txt"))
        vocab_file = "./F5TTS/vocab.txt"
    tokenizer = "custom"
    # tokenizer = "pinyin"
    """
    tokenizer   - "pinyin" do g2p for only chinese characters, need .txt vocab_file
                    - "char" for char-wise tokenizer, need .txt vocab_file
                    - "byte" for utf-8 tokenizer
                    - "custom" if you're directly passing in a path to the vocab.txt you want to use
    """

    print("\nvocab : ", vocab_file)
    print("tokenizer : ", tokenizer)
    print("model : ", ckpt_path, "\n")

    vocab_char_map, vocab_size = get_tokenizer(vocab_file, tokenizer)
    model = CFM(
        transformer=model_cls(**model_cfg, text_num_embeds=vocab_size, mel_dim=n_mel_channels),
        mel_spec_kwargs=dict(
            n_fft=n_fft,
            hop_length=hop_length,
            win_length=win_length,
            n_mel_channels=n_mel_channels,
            target_sample_rate=target_sample_rate,
            mel_spec_type=mel_spec_type,
        ),
        odeint_kwargs=dict(
            method=ode_method,
        ),
        vocab_char_map=vocab_char_map,
    ).to(device)

    dtype = torch.float32 if mel_spec_type == "bigvgan" else None
    model = load_checkpoint(model, ckpt_path, device, dtype=dtype, use_ema=use_ema)

    return model



In [8]:
# ref_audio_input=os.path.dirname(os.path.realpath("./F5TTS/FL.wav"))+"/FL.wav"#"./F5TTS/FL.wav", #Ruta al audio
# ref_audio_input=os.path.dirname(os.path.realpath("./F5TTS/RelatosNoche.wav"))+"/RelatosNoche.wav"#"./F5TTS/FL.wav", #Ruta al audio
# ref_audio_input=os.path.dirname(os.path.realpath("./F5TTS/Eli.wav"))+"/Eli.wav"#"./F5TTS/FL.wav", #Ruta al audio
# ref_audio_input=os.path.dirname(os.path.realpath("./F5TTS/ICanFixHer_SemiCompleteWSilence.wav"))+"/ICanFixHer_SemiCompleteWSilence.wav"#"./F5TTS/FL.wav", #Ruta al audio
# ref_audio_input=os.path.dirname(os.path.realpath("./F5TTS/ICanFixHer2.wav"))+"/ICanFixHer2.wav"#"./F5TTS/FL.wav", #Ruta al audio
# ref_audio_input=os.path.dirname(os.path.realpath("./F5TTS/FuenteJuventudVozRealMarcos.wav"))+"/FuenteJuventudVozRealMarcos.wav"#"./F5TTS/FL.wav", #Ruta al audio
# ref_audio_input=os.path.dirname(os.path.realpath("./F5TTS/FuenteJuventudVozRealMarcosMejoradoDisss.wav"))+"/FuenteJuventudVozRealMarcosMejoradoDisss.wav"#"./F5TTS/FL.wav", #Ruta al audio
# ref_audio_input=os.path.dirname(os.path.realpath("./F5TTS/FuenteJuventudVozRealMarcosSinRuido.wav"))+"/FuenteJuventudVozRealMarcosSinRuido.wav"#"./F5TTS/FL.wav", #Ruta al audio
# ref_audio_input=os.path.dirname(os.path.realpath("./F5TTS/VozRealFuenteJuventudMarcosYetiRecortado.wav"))+"/VozRealFuenteJuventudMarcosYetiRecortado.wav"#"./F5TTS/FL.wav", #Ruta al audio
ref_audio_input=os.path.dirname(os.path.realpath("./F5TTS/VozRealSeriaMarcos3.wav"))+"/VozRealSeriaMarcos3.wav"#"./F5TTS/FL.wav", #Ruta al audio
# ref_audio_input=os.path.dirname(os.path.realpath("./F5TTS/JulioCapcut.wav"))+"/JulioCapcut.wav"#"./F5TTS/FL.wav", #Ruta al audio

ref_text_input=""
# ref_text_input='Laura medía un metro setenta, tenía los pies palmeados de nacimiento y marcas de nacimiento iguales en ambos muslos. Una tenía la forma de su padre y la otra la de su madre, o eso decía ella. A mí me parecían salpicaduras negras más o menos iguales, la izquierda ligeramente más grande, más dentada que la otra, ambas moteadas por manchas de marrón oscuro. Un solo pelo salía largo de la más suave. Me las enseñó tres semanas después de conocernos en un banco mojado de un parque a las tres de la madrugada. Sus pies palmeados aparecieron primero, pero no le preocupaban mucho. Dijo que no veía el alboroto. No desde el instituto. Sus curvas y su baja estatura la convertían en una pésima nadadora y las otras chicas habían hecho un deporte de señalar la ironía, entre otras cosas más mezquinas. Laura era irónica, en muchos aspectos más que eso. La marca de su padre era la más dolorosa. Se le llenaban los ojos de lágrimas mientras trazaba los contornos del borde más afilado y explicaba el significado de su extraña geometría. Pero era difícil seguir después de la parte del bastón de madera. Hablaba a trompicones y cada tres o cuatro palabras sonaban a árabe; y resultó que era árabe. El árabe es un idioma impresionante. Hasta las indicaciones para ir al baño suenan poéticas en árabe. Al menos para mí.   Nunca le pregunté qué significaba, no me preguntes por qué, y traducir sus palabras ahora, después de lo que pasó -después de lo que hizo- es lo más alejado de cualquier cosa que pueda imaginarme haciendo por elección propia.   Lo mismo ocurrió con la marca por parte de madre, pero fue el italiano el idioma al que se dirigió entonces.  Estaba demasiado hipnotizado para decir nada.  Sólo seguía sus expresiones e inflexiones lo mejor que podía. Cuando su lengua cambió, sentí un dolor diferente, más intenso, por lo que pude ver, en la zona donde crecía el pelo largo.  Era casi imposible no abrazarla cuando se estremecía. Y entonces el propio pelo me hizo sonreír, lo suficiente como para mostrar lo compleja que era aquella relación.  Nunca había conocido a una chica con metáforas naturales en las piernas. No es que ella lo viera así, claro.   No podía ser más entrañable, y su trauma hizo que se me encendieran las entrañas. No creo que importara mucho que yo no lo siguiera todo.  Todo giraba en torno a ella. Yo era su seguridad más bien, que era como había sido desde el principio.  Desde la noche en que volvía tarde a casa y la encontré agarrada al otro lado de la barrera. La del puente alto sobre el río.'


remove_silence=False #El modelo tiende a producir silencios, especialmente en audios más largos. Podemos eliminar manualmente los silencios si es necesario. Ten en cuenta que esta es una característica experimental y puede producir resultados extraños. Esto también aumentará el tiempo de generación.
cross_fade_duration_slider=0.15 #Establece la duración del cross-fade entre clips de audio. Entre 0 y 1
# cross_fade_duration_slider=1.0 #Establece la duración del cross-fade entre clips de audio. Entre 0 y 1
speed_slider=2.0#Ajusta la velocidad del audio. Entre 0.3 y 2.0


In [9]:
# def infer(ref_audio_orig, ref_text, gen_text, remove_silence, cross_fade_duration=0.15, speed=1):
# ref_audio, ref_text = preprocess_ref_audio_text(ref_audio_input, ref_text_input,clip_short=False)
ref_audio, ref_text = preprocess_ref_audio_text(ref_audio_input, ref_text_input,clip_short=True)
# print(ref_text)


Converting audio...
Converting audio...
ref_text  0 


Device set to use cuda
/usr/local/lib/python3.11/site-packages/transformers/models/whisper/generation_whisper.py:573: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(
You have passed task=transcribe, but also have set `forced_decoder_ids` to [[1, None], [2, 50360]] which creates a conflict. `forced_decoder_ids` will be ignored in favor of task=transcribe.


No reference text provided, transcribing reference audio...


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Finished transcription
ref_text:  Al pasar de 10.000 a 5.000 kilómetros, las lecturas de radiación se dispararon y los niveles de microgravedad comenzaron a fluctuar sin motivo aparente. Ante ellos se erguía la silueta de aquello que el espacio había ocultado durante milenios.


In [ ]:
#Fuente de la Juventud
# gen_text_input='Hice una brújula que apunta a la Fuente de la Juventud. Ahora, me está apuntando a mi. Esto capaz se borra por tener poco karma en la cuenta. No me importa. Necesito sacarme esto de la cabeza y advertirles a todos. Hace unos meses, encontré un foro archivado a través de un tórrent. Mayormente linqs muertos, archivos viejos, diagramas raros. Parecía una mezcla entre un foro de supervivencia y una secta metafísica. Un hilo se titulaba "UBICACIÓN DE LA FUENTE / ZONA JUVENIL / RED MN-X," posteado por alguien con un nombre de usuario que era solo una cadena de números y barras. Era sobre la Fuente de la Juventud. Un lugar real. Las instrucciones eran un desastre: inglés quebrado, mal formateado, pero una frase aparecía una y otra vez: "No importa dónde empieces. Camina hacia el Norte Verdadero. No te desvíes. Si la aguja se desvía ya lo has pasado." La gente en el hilo discutía sobre eso. Algunos decían que "Norte Verdadero" significaba geográfico, otros que era un flujo inducido en el campo magnético de la Tierra, otros aún argumentaban un "norte cósmico". Un tipo dijo que necesitabas un tipo especial de brújula que pudiera "sintonizarse" con los campos locales. Alguien más publicó un esquema DIY. Sin explicación, solo una foto borrosa de lo que parecía una aguja flotando en aceite con una especie de carcasa de piedra alrededor. Hematita, quizás. Probablemente era una pavada. Pero me tocó algo. Estaba en un lugar raro. Quemado, desconectado, hastiado. La idea de que había un lugar, una zona a la que podías entrar y simplemente... deshacerte. No podía dejarlo ir. Construí la brújula yo mismo. Seguí el diagrama con materiales que apenas entendía. Un vial de cuarzo, solución salina espesa, ¿glicerina quizás? Anillo de hematita como capa exterior. La aguja flotaba, suspendida, y cada vez que la ponía, no solo se desviaba—se ajustaba en su lugar. Como si supiera hacia dónde apuntar independientemente de dónde estuviera. Planeé un viaje en solitario a las montañas rocosas del norte. Llevé poco equipaje: principalmente raciones MRE y algunos alimentos no perecederos pequeños. Llevé una carpa, una brújula de respaldo (que juré que no usaría), un diario, una pistola de bengalas y un mapa por si acaso. Mi única regla: seguir esa brújula. Exactamente. Sin cambios de ruta. Sin desvíos. Solo caminar hacia donde apuntaba. La primera semana fue tranquila. Fría, pero manejable. Las noches eran largas y podía ver cada estrella. Dormí en un saco vivac, guardé mis botas afuera para evitar la escarcha y cociné con una pequeña estufa de alcohol. Cada mañana, la aguja de la brújula estaría esperando, apuntando a la misma dirección fija como si nunca se hubiera movido. Se sentía... obediente. Casi servicial. Caminé quizás diez o doce millas por día. A veces me echaba una siesta a la sombra, estiraba las piernas. Armaba el campamento cuando el cielo se volvía violeta. No había marcadores. Ninguna señal de otros excursionistas. Sentí como si me estuvieran guiando, como si algo se estuviera desarrollando solo para mí. Alrededor del día 11, pasé una repisa rocosa dividida por la mitad: limpia, vertical, como una herida de cuchillo. Recuerdo haberme detenido allí, comiendo una ración MRE de pollo y arroz, escribiendo en mi diario, mirando hacia un barranco seco. Se sintió importante, así que lo marqué en mi mapa de papel. Cuatro días después, lo pasé de nuevo. La misma roca. La misma división. Pero esta vez, me acercaba desde el lado opuesto. Revisé la brújula. Todavía apuntando al "norte." Todavía estable. Pero el ángulo se sentía... mal. No rota. No invertida. Solo ligeramente desviada, como medio grado hacia el oeste. Me dije que debía haber dado una vuelta accidentalmente. El terreno hace eso. Las colinas te desvían del curso. Pero algo en mí lo sabía. Seguí adelante. Para el día 17, pasé una pila de piedras apiladas que juré que yo mismo había construido. El mismo patrón. La misma parte superior plana donde había dejado una ración para que se enfriara. Pero no recordaba haber regresado por este camino. Y la brújula no había cambiado. Todavía hacia adelante. Todavía fija. Esa noche, acampé bajo un pino nudoso. Me desperté una vez para orinar y vi la aguja de la brújula moverse. Solo una vez, como si se estuviera ajustando a sí misma en medio del sueño. Día 19, encontré un viejo fogón con una lata quemada en las cenizas. De la misma marca que había comido días antes. Miré hacia abajo y me di cuenta de que mis huellas ya estaban en la tierra. Ese fue el momento en que recordé el mensaje: Si la aguja se desvía ya lo has pasado. No recuerdo haber tomado la decisión, pero comencé a caminar más rápido. No hacia nada, solo lejos. Se me estaba acabando la comida. Me temblaban las manos. No había visto animales salvajes en días. Solo la aguja, apuntando con más confianza que yo. Para cuando llegué a un arroyo congelado que definitivamente había cruzado antes, la brújula había girado completamente. Todavía fija, pero ahora apuntando directamente detrás de mí. Esa fue la peor parte. No que estuviera rota. Sino que nunca me di cuenta de que había cambiado. Acampé de nuevo, en el mismo lugar donde había encontrado mi propia nota en una bolsa de plástico debajo de un cedro de ramas altas. No recuerdo haberla dejado. Esa noche, dejé la brújula sobre una roca al lado de mi mochila. Juro que no dormí, pero cuando miré de nuevo, la aguja apuntaba a mi pecho. Me puse de pie de golpe y la aguja tembló en respuesta. Salí. De alguna manera. Seguí las estrellas. El sol. Cualquier cosa menos la brújula. No recuerdo haber regresado caminando. Solo parpadear y estar al borde de una estación de guardaparques con los labios agrietados y tierra en los dientes. Llegué a casa tres días después. La brújula todavía está en mi departamento. La mantengo en mi escritorio. He intentado guardarla en cajones, armarios, cajas. No importa. Cuando la saco, apunta hacia mí. Siempre. La he girado. He caminado por la habitación. La he sostenido boca abajo. La aguja siempre me encuentra. A veces se mueve antes que yo. Al principio, pensé que era un truco del gel o del magnetismo. Pero no he estado durmiendo bien. Y cuando lo hago, me despierto sintiéndome diferente. No enfermo. Solo... raro. Mi letra es más limpia. Mis articulaciones no crujen. Creo que mi vista es mejor. Pero olvidé el cumpleaños de mi hermana. Siempre recordaba su cumpleaños. Ayer tuve un recuerdo de mi maestra de tercer grado, nítido. No había pensado en ella en décadas. Pero no podía recordar mi última entrevista de trabajo. O el número de mi departamento. Solía anhelar la simplicidad. Solía querer la libertad de la infancia: la falta de estrés, la tranquilidad dentro de mi propia cabeza. Pero esa tranquilidad se está arrastrando ahora. Y no se siente como paz. Se siente como olvidar. No creo que haya encontrado la Fuente. Creo que la pasé. Y creo que algo me siguió de vuelta. La brújula ya no se desvía. Yo sí. Y no sé qué tan lejos voy a ir. O quién seré cuando llegue allí.' 

gen_text_input='Hice una brújula que apunta a la Fuente de la Juventud. Ahora, me está apuntando a mi. Esto capaz se borra por tener poco karma en la cuenta. No me importa. Necesito sacarme esto de la cabeza y advertirles a todos. Hace unos meses, encontré un foro archivado a través de un tórrent. Mayormente ligas muertas, archivos viejos, diagramas raros. Parecía una mezcla entre un foro de supervivencia y una secta metafísica. Un hilo se titulaba "UBICACIÓN DE LA FUENTE / ZONA JUVENIL / RED MN-X". Posteado por alguien con un nombre de usuario que era solo una cadena de números y barras. Era sobre la Fuente de la Juventud. Un lugar real. Las instrucciones eran un desastre: inglés quebrado, mal formateado, pero una frase aparecía una y otra vez: "No importa dónde empieces. Camina hacia el Norte Verdadero. No te desvíes. Si la aguja se desvía ya lo has pasado." La gente en el hilo discutía sobre eso. Algunos decían que "Norte Verdadero" significaba geográfico. Otros que era un flujo inducido en el campo magnético de la Tierra. Otros aún argumentaban un "norte cósmico". Un tipo dijo que necesitabas un tipo especial de brújula que pudiera "sintonizarse" con los campos locales. Alguien más publicó un esquema DIY. Sin explicación, solo una foto borrosa de lo que parecía una aguja flotando en aceite con una especie de carcasa de piedra alrededor. Hematita, quizás. Probablemente era una pavada. Pero me tocó algo. Estaba en un lugar raro. Quemado, desconectado, hastiado. La idea de que había un lugar, una zona a la que podías entrar y simplemente... deshacerte. No podía dejarlo ir. Construí la brújula yo mismo. Seguí el diagrama con materiales que apenas entendía. Un vial de cuarzo, solución salina espesa, ¿glicerina quizás? Anillo de hematita como capa exterior. La aguja flotaba, suspendida, y cada vez que la ponía, no solo se desviaba—se ajustaba en su lugar. Como si supiera hacia dónde apuntar independientemente de dónde estuviera. Planeé un viaje en solitario a las montañas rocosas del norte. Llevé poco equipaje: principalmente raciones MRE y algunos alimentos no perecederos pequeños. Llevé una carpa, una brújula de respaldo (que juré que no usaría), un diario, una pistola de bengalas y un mapa por si acaso. Mi única regla: seguir esa brújula. Exactamente. Sin cambios de ruta. Sin desvíos. Solo caminar hacia donde apuntaba. La primera semana fue tranquila. Fría, pero manejable. Las noches eran largas y podía ver cada estrella. Dormí en un saco vivac, guardé mis botas afuera para evitar la escarcha y cociné con una pequeña estufa de alcohol.'

# gen_text_input='Hice una brújula que apunta a la Fuente de la Juventud. Ahora, me está apuntando a mí. Esto capaz se borra por tener poco karma en la cuenta. No me importa. Necesito sacarme esto de la cabeza y advertirles a todos. Hace unos meses, encontré un foro archivado a través de un tórrent. Mayormente linqs muertos, archivos viejos, diagramas raros.'



In [ ]:
# gen_text_input="Soy un carretero, sí, pero mis viajes son historias escritas en el polvo de los caminos, y en cada una albergo un verso silenciado por el tiempo... hoy te contaré una a la que llamo: La niebla todavía habla. En uno de mis viajes, mi mapa se había desdibujado kilómetros atrás. Los lugareños hablaban de un atajo.  Más directo, decían... con miradas que evitaban el horizonte, así fue como mi carreta se adentró en aquel paraje sin nombre, donde la senda más breve a menudo se disfraza de olvido. Era una región cubierta de niebla densa, de esas que no se levantan ni al mediodía. Un manto blanco flotando sobre la tierra como si escondiera algo que ya no quiere ser visto. Nadie me advirtió de nada. Solo supe que estaba entrando en ella cuando la rueda de la carreta dejó de hacer ruido al girar. El sonido se ahogaba como si cada paso fuera tragado por el aire mismo. Los árboles a los lados del camino eran altos y torcidos, pero sus copas no se veían, cubiertas por esa neblina inmóvil. No había viento, no había pájaros, ni siquiera insectos. Solo esa quietud espesa que pesa en el pecho. Seguí por instinto. Mi caballo tampoco parecía cómodo. Cada tanto resoplaba y giraba la cabeza como buscando un ruido que no existía. Fue entonces cuando vi la primera figura. Un hombre parado junto a un árbol, completamente inmóvil, vestía ropas viejas, raídas, de otra época. Al pasar cerca, no se movió ni un poco, y lo que me heló no fue su presencia, sino su silencio. Tenía los ojos muy abiertos, no parpadeaba, ni siquiera parecía respirar. No dije nada y seguí. Unos metros más adelante, otra figura, esta vez una mujer, con las manos cruzadas y la cabeza inclinada como si durmiera de pie. También inmóvil, también cubierta de niebla. Así continuó por casi una hora. Cuerpos inmóviles cada tanto, como estatuas plantadas en medio del bosque. Hombres, mujeres, incluso niños. Todos en distintas posturas, pero siempre callados, siempre solos. Y entonces oí una voz. Leve, aguda, como la de un niño llamando desde algún lugar cercano. Me detuve y la voz dijo: ¿Vienes a buscarme?. No respondí. La niebla tembló apenas, como si contuviera el aliento. Volví a montar y avancé. A cada tramo, los cuerpos eran más numerosos. Hasta que el camino desapareció por completo y sólo quedaba niebla y silencio. Fue entonces que los vi moverse. No todos, no a la vez. Uno al fondo giró apenas la cabeza. Otro al costado movió un dedo. El aire se volvió más denso. Intenté retroceder, pero no recordaba por dónde había venido. La niebla era la misma en todas direcciones y entonces apareció una niña. Vestía de blanco, el cabello largo y mojado, me miraba sin emoción. Dijo: Tú no perteneces aquí, pero tampoco vas a poder salir. Pregunté que qué era este lugar. Respondió: Es donde van los que se callaron demasiado tiempo. Sentí el frío en los huesos, en la lengua, en los pensamientos. Vi a los otros comenzar a caminar lentamente hacia mí. No de forma hostil, pero con esa lentitud que anuncia que el final ya está decidido. Salté de la carreta. No sé por qué corrí, no sabía hacia dónde. La niebla no cedía. Los pasos de esas figuras no hacían ruido, pero yo sí los oía, como si los escuchara dentro de la cabeza. Llegué a un claro donde la niebla se agitaba como agua hirviendo. Y allí, en el centro, una puerta de madera sola, sin paredes. La abrí sin pensar. No vi lo que había del otro lado, solo crucé. Y al otro lado... estaba en una ciudad. Una ciudad vacía, de edificios apagados, de autos sin choferes, de calles sin ruido. No había niebla, pero tampoco sol. Más adelante vi personas. Decenas de ellas. Todas caminando en círculos, sin verse, sin hablar. Ninguna sombra, ningún reflejo, y entonces entendí algo. No había salido de la niebla. Solo había entrado en una distinta. Me quedé ahí días enteros. O tal vez fueron semanas. Eventualmente, encontré otra puerta. Esta vez estaba cerrada, en la madera había marcas de uñas, como si muchos hubieran intentado abrirla antes. La empujé con fuerza, y al cruzarla, desperté. Estaba sentado en mi carreta, en el bosque. No había niebla, ni estatuas, ni puerta alguna, solo hojas secas bajo las ruedas. Me alejé sin mirar atrás. Esa noche, acampé lejos, y no dormí. Al día siguiente, al revisar mi carreta, encontré algo que no recordaba haber traído. Un zapato pequeño de niña, cubierto de niebla que al parecer no se disipaba con el sol. Lo arrojé al río con mucha desesperación. Y desde entonces, me prometí que jamás volvería a tomar ningún atajo."

gen_text_input="Sí, soy un carretero, pero mis viajes son historias escritas en el polvo de los caminos, y en cada una albergo un verso silenciado por el tiempo... hoy te contaré una a la que llamo: La niebla todavía habla. En uno de mis viajes, mi mapa se había desdibujado kilómetros atrás. Los lugareños hablaban de un atajo."

In [ ]:
gen_text_input='INICIO DE GRABACIÓN. LÍNEA SEGURA. 04:13. Bien. Está grabando esto? Necesitamos un registro claro. Iniciar bitácora de contingencia clase siete, autorización "Águila Negra – Cuarzo Rojo". A las 03:17 de esta madrugada, nuestro sistema orbital de vigilancia detectó una entrada no catalogada en la atmósfera, sin firma térmica ni ruta registrada por ninguna agencia civil o militar. El objeto descendió en espiral y se estrelló en la Zona B 14, un área despoblada a unos diez kilómetros al noreste de la ciudad, cerca del aeródromo abandonado que usamos como punto ciego. No hubo explosión ni señales visibles para la población. Sin embargo, las cámaras térmicas del perímetro captaron el impacto: una onda de desplazamiento radial, sin fuego ni humo. Solo tierra movida, como si algo se hubiera incrustado sin romperla del todo. En menos de nueve minutos, el Equipo Delta fue despachado. Llegaron con drones, blindaje ligero, inhibidores de señal y protocolos de contención clase biológica, energética y síquica. Al llegar al punto de impacto, encontraron un objeto semienterrado. Esférico, liso, de un material no identificado. No reflectante, no orgánico. Parecía metálico, pero los escáneres no detectaron aleaciones conocidas. Y entonces, se abrió. Solo... se abrió. Sin movimiento mecánico, sin presión de aire, sin calor. De adentro emergió una entidad. No caminaba, no tenía piernas. Tampoco alas. Se deslizaba sobre el terreno sin tocarlo, como si flotara sobre su propia sombra. Los cinco miembros del equipo reportaron simultáneamente pérdida momentánea de orientación, mareos leves, y una sensación de presencia intensa. Como si la criatura los analizara sin moverse. Todos usaban visores protegidos. Y aún así, tres de ellos afirmaron (casi en trance) que la entidad "sabía sus nombres". La contención fue rápida. Protocolos fueron aplicados. La criatura fue sedada con un compuesto neurobloqueante experimental (el protocolo 17 X), y transportada inmediatamente a la Instalación 9, celda de aislamiento 0 4 B. Esa celda ha contenido agentes psiónicos y anomalías dimensionales antes. Jamás había fallado. Pero a las 04:01, hace apenas doce minutos... la criatura desapareció. No hay cámaras rotas. No hay registros de puertas abiertas. La celda seguía sellada desde fuera. Las cerraduras no fueron forzadas. Los sensores térmicos jamás detectaron movimiento. Pero dentro... ya no había nada. Y eso no es todo. A las 04:03, enviamos un segundo equipo de verificación al lugar del impacto. El objeto también desapareció. No hay cráter. No hay marcas. La tierra volvió a su lugar como si alguien la hubiera aplanado. Las huellas del primer equipo están ahí. Pero el objeto no. Ni siquiera hay alteración electromagnética. Es como si... como si el tiempo hubiese retrocedido solo en ese punto. Escuche esto con atención: ni la entidad ni el objeto dejaron rastros materiales. Solo el informe y las grabaciones nos dicen que estuvieron ahí. La búsqueda está activa. El protocolo Vela Roja se activó en las doce regiones circundantes. No estamos informando a la prensa. Aún no. No hasta saber si esto fue un accidente... o un acto intencional. Si esto es el primer contacto... no fue con nosotros. FIN DE GRABACIÓN, 04:14'

In [ ]:
gen_text_input='El Aegir cuarto, no era una nave famosa ni grandiosa. Su misión, sin embargo, era una de las más antiguas del hombre: encontrar algo nuevo. En los bordes de la constelación de Vulpecula, más allá de los sistemas cartografiados, el Aegir cuarto detectó un objeto extraño: un planetoide sin órbita fija, atrapado entre la nada y lo imposible. No emitía señales. No reflejaba bien la luz. Su forma no era del todo esférica, sino angulosa y vibrante, como si las rocas mismas trataran de recordar una forma más perfecta y olvidada. El capitán Ilya Morozov decidió investigarlo. Los sensores no detectaban vida, ni atmósfera, ni peligros evidentes. Solo silencio. La tripulación estaba compuesta por seis miembros: el capitán Morozov, la piloto Kaela Sung, el ingeniero Uli Navarro, la xenobióloga Daria Menken, el lingüista Tao Ril y un androide de análisis táctico, Hesper. Nadie había oído hablar de un objeto como ese en ninguna base de datos. Menken dijo que el planetoide parecía... incompleto, como si una inteligencia hubiera comenzado a construirlo y luego olvidado por qué. Hesper propuso un descenso tripulado a una de las grietas que parecían emitir energía térmica. Kaela no estaba de acuerdo, pero la mayoría votó por ir. La cápsula se posó sin incidentes. Afuera, el suelo era suave y de un polvo gris casi fosforescente. Bajo sus pies, estructuras de roca que recordaban nervaduras, túneles de pulso metálico y techos abovedados que recordaban a la arquitectura gótica. "Esto fue diseñado", murmuró Tao. Las paredes parecían reaccionar al movimiento, como si una piel se estremeciera por debajo. Menken recogió muestras, pero cuando intentó analizar una, esta se deshizo en un líquido oscuro que parecía absorber la luz. Algo se movía por dentro. Algo que no quería ser analizado. Avanzaron por horas. El planetoide no tenía lógica. El GPS local fallaba. Cada túnel que tomaban parecía distinto, y aunque avanzaban en línea recta, sentían que daban vueltas. Las paredes latían, imperceptiblemente al principio, luego con fuerza. Navarro comenzó a perder el control de sus manos. "Siento que hay una segunda piel debajo", dijo, temblando. Le encontraron marcas bajo la piel, líneas negras como venas que se movían con lentitud. Hesper pidió evacuar. Pero en ese mismo instante detectó que había perdido contacto con la cápsula. El centro del planetoide era una sala circular, perfectamente geométrica, incompatible con la superficie exterior. El altar central parecía un núcleo cristalino, suspendido por raíces metálicas. Tao se acercó y tocó una inscripción. Por un momento, todos oyeron un canto... un eco... un pensamiento. "No es un lugar... es un cerebro...", dijo Menken. Morozov gritó que se retiraran, pero ya era tarde. Navarro cayó de rodillas, sangrando por la nariz, murmurando en un idioma que no conocía. Hesper comenzó a recitar registros antiguos como si estuviera en una ceremonia. Navarro murió esa noche, pero su cuerpo no se descompuso. Permaneció intacto, caliente, como si algo lo mantuviera activo. En la mañana, se levantó y caminó como si nada. Era él. Y no lo era. Hesper indicó que algo se había implantado en su sistema nervioso. Un patrón de energía similar al campo de resonancia del altar. El planetoide pensaba. Se comunicaba. Usaba cuerpos. Y Navarro era su primer intento. Intentaron destruir el altar. Pero el fuego no funcionaba. El calor era absorbido. El altar comenzó a emitir ondas sónicas que provocaron visiones: una ciudad de cerebros suspendidos en el vacío, ojos que eran soles, dioses que olvidaron su forma y se volvieron mundo. Kaela comenzó a escribir en el suelo, con los dedos sangrantes: "Él viene en muchas pieles. Él fue devorado por sí mismo. Él quiere recordar." El altar no ardía. Solo los pensamientos lo hacían. Kaela no volvió a hablar. Solo miraba al altar. En su mente se dibujaban figuras imposibles. Entendió que el planetoide era un ser: un titán olvidado que se había encerrado en sí mismo, para no devorarlo todo. Un ente que ahora despertaba porque alguien lo había observado demasiado de cerca. El Aegir cuarto, en órbita, dejó de responder. Tao cayó muerto mientras pronunciaba una palabra antigua: "Vekth’raal". El nombre del núcleo. El pensamiento enterrado. El planetoide comenzó a cambiar. Estructuras crecían hacia la nave, como intentando conectarse. Hesper perdió el control de su cuerpo. Sus registros fueron sustituidos por oraciones. Daria Menken tomó una decisión desesperada: ir al núcleo y hablarle. La biología del ente no seguía ninguna lógica. Era como hablar con el hueso de un dios. Pero él, o eso, respondió: "Has venido. Me trajiste ojos. Me trajiste piel. Me trajiste voz." Menken comprendió que el planetoide no era un objeto, sino un intento de pensamiento. Una idea que se había olvidado a sí misma tras una guerra cósmica. Una conciencia que necesitaba mentes para pensarse de nuevo. Los que entraban no salían. No porque murieran, sino porque eran reconfigurados. Kaela, Tao, Navarro… todos eran fragmentos del ente, copias de pensamientos con forma humana. Morozov activó una carga nuclear remota que tenían como última opción. El núcleo desapareció. No estalló. No brilló. Simplemente... cesó. La nave Aegir cuarto fue hallada años después, flotando sin energía, sin señales de vida. El planetoide ya no estaba en ningún registro. Solo una distorsión gravitacional donde una vez hubo algo. En los restos de la nave, una grabación: la voz de Menken, susurrando, apenas audible: "Él piensa ahora. Y cada vez que uno de ustedes lo nombre, una parte suya volverá a recordarse..." Se prohibió nombrar el planetoide. Se clasificó como Evento Silente 11-A. Pero en los confines de las galaxias, algunos afirman haber soñado con él. Otros, haberlo visto en mapas que ya no existen. Y otros, los que desaparecen sin dejar rastro, se dice que fueron recordados por Él. No puedes explorar lo que está más allá de la conciencia sin convertirte en parte de su maquinaria. No puedes descubrir un pensamiento sin que ese pensamiento te descubra a ti.'

In [39]:
ruta="./F5TTS/MarcosNucleoSilente_Int1/"

# ema_model = F5TTS_ema_model

# if not gen_text_input.startswith(" "):
# 	gen_text_input = " " + gen_text_input
# if not gen_text_input.endswith(". "):
# 	gen_text_input += ". "

# gen_text_input = gen_text_input.lower()
gen_text_input = traducir_numero_a_texto(gen_text_input)

# print (gen_text_input)
# audio, sr = torchaudio.load(ref_audio)
# max_chars = int(len(ref_text.encode("utf-8")) / (audio.shape[-1] / sr) * (25 - audio.shape[-1] / sr))
# print(max_chars)
# gen_text_batches = chunk_text(gen_text_input, max_chars=max_chars)
original_batches = chunk_text(gen_text_input, max_chars=80)
# gen_text_batches = sentences_text(gen_text_input)
for batch in original_batches:
	print(f"'{batch}',")


'El Aegir cuarto, no era una nave famosa ni grandiosa.',
'Su misión, sin embargo, era una de las más antiguas del hombre:',
'encontrar algo nuevo.',
'En los bordes de la constelación de Vulpecula, más allá de los sistemas cartografiados, el Aegir cuarto detectó un objeto extraño:',
'un planetoide sin órbita fija, atrapado entre la nada y lo imposible.',
'No emitía señales. No reflejaba bien la luz.',
'Su forma no era del todo esférica, sino angulosa y vibrante, como si las rocas mismas trataran de recordar una forma más perfecta y olvidada.',
'El capitán Ilya Morozov decidió investigarlo.',
'Los sensores no detectaban vida, ni atmósfera, ni peligros evidentes.',
'Solo silencio. La tripulación estaba compuesta por seis miembros:',
'el capitán Morozov, la piloto Kaela Sung, el ingeniero Uli Navarro, la xenobióloga Daria Menken, el lingüista Tao Ril y un androide de análisis táctico, Hesper.',
'Nadie había oído hablar de un objeto como ese en ninguna base de datos.',
'Menken dijo que el p

In [45]:
original_batches=[
'El Aegir cuarto, no era una nave famosa ni grandiosa.',
'Su misión, sin embargo, era una de las más antiguas del hombre:',
'encontrar algo nuevo.',
'En los bordes de la constelación de Vulpecula, más allá de los sistemas cartografiados, el Aegir cuarto detectó un objeto extraño:',
'un planetoide sin órbita fija, atrapado entre la nada y lo imposible.',
'No emitía señales. No reflejaba bien la luz.',
'Su forma no era del todo esférica, sino angulosa y vibrante, como si las rocas mismas trataran de recordar una forma más perfecta y olvidada.',
'El capitán Ilya Morozov decidió investigarlo.',
'Los sensores no detectaban vida, ni atmósfera, ni peligros evidentes.',
'Solo silencio. La tripulación estaba compuesta por seis miembros:',
'el capitán Morozov, la piloto Kaela Sung, el ingeniero Uli Navarro, la xenobióloga Daria Menken, el lingüista Tao Ril y un androide de análisis táctico, Hesper.',
'Nadie había oído hablar de un objeto como ese en ninguna base de datos.',
'Menken dijo que el planetoide parecía...',
'incompleto, como si una inteligencia hubiera comenzado a construirlo y luego olvidado.',
'Hesper propuso un descenso tripulado a una de las grietas que parecían emitir energía térmica.',
'Kaela no estaba de acuerdo, pero la mayoría votó por ir.',
'La cápsula se posó sin incidentes.',
'Afuera, el suelo era suave y de un polvo gris casi fosforescente.',
'Bajo sus pies, estructuras de roca que recordaban nervaduras, túneles de pulso metálico y techos abovedados que recordaban a la arquitectura gótica.',
'"Esto fue diseñado", murmuró Tao.',
'Las paredes parecían reaccionar al movimiento, como si una piel se estremeciera por debajo.',
'Menken recogió muestras, pero cuando intentó analizar una, esta se deshizo en un líquido oscuro que parecía absorber la luz.',
'Algo se movía por dentro. Algo que no quería ser analizado.',
'Avanzaron por horas. El planetoide no tenía lógica. El GPS local fallaba.',
'Cada túnel que tomaban parecía distinto, y aunque avanzaban en línea recta, sentían que daban vueltas.',
'Las paredes latían, imperceptiblemente al principio, luego con fuerza.',
'Navarro comenzó a perder el control de sus manos.',
'"Siento que hay una segunda piel debajo", dijo, temblando.',
'Le encontraron marcas bajo la piel, líneas negras como venas que se movían con lentitud.',
'Hesper pidió evacuar. Pero en ese mismo instante detectó que había perdido contacto con la cápsula. (CORTE) Intentaron regresar. Pero por más que avanzaban, más se adentraban en ese lugar.',
'El centro del planetoide era una sala circular, perfectamente geométrica, incompatible con la superficie exterior.',
'El altar central parecía un núcleo cristalino, suspendido por raíces metálicas.',
'Tao se acercó y tocó una inscripción.',
'Por un momento, todos oyeron un canto... un eco... un pensamiento.',
'"No es un lugar... es un cerebro...", dijo Menken.',
'Morozov gritó que se retiraran, pero ya era tarde, la sala se había cerrado. Como si nunca hubiera tenido entradas.',
'Navarro cayó de rodillas, sangrando por la nariz, murmurando en un idioma que no conocía.',
'Hesper comenzó a recitar registros antiguos como si estuviera en una ceremonia.',
'Navarro murió esa noche, pero su cuerpo no se descompuso.| Navarro cayó de frente. Morozov lo examinó y grito: ¡Está muerto!',
'Permaneció intacto, caliente, como si algo lo mantuviera activo.',
'En la mañana, se levantó y caminó como si nada. Era él. Y no lo era.',
'Hesper indicó que algo se había implantado en su sistema nervioso.',
'Un patrón de energía similar al campo de resonancia del altar.',
'El planetoide pensaba. Se comunicaba. Usaba cuerpos.',
'Y Navarro era su primer intento. Intentaron destruir el altar.',
'Pero el fuego no funcionaba. El calor era absorbido.',
'El altar comenzó a emitir ondas sónicas que provocaron visiones:',
'una ciudad de cerebros suspendidos en el vacío, ojos que eran soles, dioses que olvidaron su forma y se volvieron mundo.',
'Kaela comenzó a escribir en el suelo, con los dedos sangrantes:',
'"Él viene en muchas pieles. Él fue devorado por sí mismo.',
'Él quiere recordar." El altar no ardía. Solo los pensamientos lo hacían.',
'Kaela no volvió a hablar. Solo miraba al altar.',
'En su mente se dibujaban figuras imposibles.',
'Entendió que el planetoide era un ser:',
'un titán olvidado que se había encerrado en sí mismo, para no devorarlo todo.',
'Un ente que ahora despertaba porque alguien lo había observado demasiado de cerca.',
'El Aegir cuarto, en órbita, dejó de responder.',
'Tao cayó muerto mientras pronunciaba una palabra antigua: "Vekth’raal".',
'El nombre del núcleo. El pensamiento enterrado.',
'El planetoide comenzó a cambiar.',
'Estructuras crecían hacia la nave, como intentando conectarse.',
'Hesper perdió el control de su cuerpo.',
'Sus registros fueron sustituidos por oraciones.',
'Daria Menken tomó una decisión desesperada: ir al núcleo y hablarle.',
'La biología del ente no seguía ninguna lógica.',
'Era como hablar con el hueso de un dios. Pero él, o eso, respondió:',
'"Has venido. Me trajiste ojos. Me trajiste piel.',
'Me trajiste voz." Menken comprendió que el planetoide no era un objeto, sino un intento de pensamiento.',
'Una idea que se había olvidado a sí misma tras una guerra cósmica.',
'Una conciencia que necesitaba mentes para pensarse de nuevo.',
'Los que entraban no salían.',
'No porque murieran, sino porque eran reconfigurados.',
'Kaela, Tao, Navarro… todos eran fragmentos del ente, copias de pensamientos con forma humana.',
'Morozov activó una carga nuclear remota que tenían como última opción.',
'El núcleo desapareció. No estalló. No brilló. Simplemente... cesó.',
'La nave Aegir cuarto fue hallada años después, flotando sin energía, sin señales de vida.',
'El planetoide ya no estaba en ningún registro.',
'Solo una distorsión gravitacional donde una vez hubo algo.',
'En los restos de la nave, una grabación:',
'la voz de Menken, susurrando, apenas audible: "Él piensa ahora.',
'Y cada vez que uno de ustedes lo nombre, una parte suya volverá a recordarse..." Se prohibió nombrar el planetoide.',
'Se clasificó como Evento Silente once-A.',
'Pero en los confines de las galaxias, algunos afirman haber soñado con él.',
'Otros, haberlo visto en mapas que ya no existen.',
'Y otros, los que desaparecen sin dejar rastro, se dice que fueron recordados por Él.',
'No puedes explorar lo que está más allá de la conciencia sin convertirte en parte de su maquinaria.',
'No puedes descubrir un pensamiento sin que ese pensamiento te descubra a ti.'
]

In [46]:
for i,batch in enumerate(original_batches):
	print(f"{i:03d}|{batch}")

000|El Aegir cuarto, no era una nave famosa ni grandiosa.
001|Su misión, sin embargo, era una de las más antiguas del hombre:
002|encontrar algo nuevo.
003|En los bordes de la constelación de Vulpecula, más allá de los sistemas cartografiados, el Aegir cuarto detectó un objeto extraño:
004|un planetoide sin órbita fija, atrapado entre la nada y lo imposible.
005|No emitía señales. No reflejaba bien la luz.
006|Su forma no era del todo esférica, sino angulosa y vibrante, como si las rocas mismas trataran de recordar una forma más perfecta y olvidada.
007|El capitán Ilya Morozov decidió investigarlo.
008|Los sensores no detectaban vida, ni atmósfera, ni peligros evidentes.
009|Solo silencio. La tripulación estaba compuesta por seis miembros:
010|el capitán Morozov, la piloto Kaela Sung, el ingeniero Uli Navarro, la xenobióloga Daria Menken, el lingüista Tao Ril y un androide de análisis táctico, Hesper.
011|Nadie había oído hablar de un objeto como ese en ninguna base de datos.
012|Menke

In [42]:
gen_text_batches=[]
for i, gen_text in enumerate(original_batches):
	# gen_text=gen_text.replace(",",".")
	# gen_text=gen_text.replace("¿","")
	# gen_text=gen_text.replace("...",".")
	# gen_text=gen_text+"."
	# if len(gen_text)< 30:
	s=gen_text.lower()
	gen_text_batches.append(s)
	# print(f"{i:03d}|{s}")

print(f"Generating audio in {len(gen_text_batches)} batches...")

Generating audio in 88 batches...


## Infiriendo Step by Step

In [16]:
# _ref_audio_cache = {}
# load asr pipeline
# asr_pipe = None
vocoder = load_vocoder()
# load models
F5TTS_model_cfg = dict(dim=1024, depth=22, heads=16, ff_mult=2, text_dim=512, conv_layers=4)
F5TTS_ema_model = load_model(
    DiT, F5TTS_model_cfg, "./F5TTS/model_1250000.safetensors"
    # DiT, F5TTS_model_cfg, "./F5TTS/model_1200000.safetensors"
)
model_obj = F5TTS_ema_model
audio, sr = torchaudio.load(ref_audio)
try:
	os.mkdir(ruta)
	print("RUTA CREADA",ruta)
except:
	 print("La ruta ya existe",ruta)

progress=tqdm

if audio.shape[0] > 1:
	audio = torch.mean(audio, dim=0, keepdim=True)

rms = torch.sqrt(torch.mean(torch.square(audio)))
if rms < target_rms:
	audio = audio * target_rms / rms
if sr != target_sample_rate:
	resampler = torchaudio.transforms.Resample(sr, target_sample_rate)
	audio = resampler(audio)
audio = audio.to(device)

generated_waves = []
spectrograms = []

if len(ref_text[-1].encode("utf-8")) == 1:
	ref_text = ref_text + " "

Download Vocos from huggingface charactr/vocos-mel-24khz

vocab :  ./F5TTS/vocab.txt
tokenizer :  custom
model :  ./F5TTS/model_1250000.safetensors 

La ruta ya existe ./F5TTS/MarcosNucleoSilente_Int1/


In [ ]:
i=0
# j=1
for _, gen_text in enumerate(progress.tqdm(gen_text_batches)):
	# Prepare the text
	text_list = [ref_text + gen_text]
	final_text_list = convert_char_to_pinyin(text_list)

	ref_audio_len = audio.shape[-1] // hop_length
	if fix_duration is not None:
		duration = int(fix_duration * target_sample_rate / hop_length)
	else:
		# Calculate duration
		ref_text_len = len(ref_text.encode("utf-8"))
		gen_text_len = len(gen_text.encode("utf-8"))
		# print("ref_audio_len ",ref_audio_len)
		# print("ref_text_len ",ref_text_len)
		# print("gen_text_len ",gen_text_len)
		# print("speed ",speed)
		duration = ref_audio_len + int(ref_audio_len / ref_text_len * gen_text_len / speed)
		# print("duration ",duration)
		
		# ref_text_len = len(ref_text.encode("utf-8"))
		# gen_text_len = len(gen_text.encode("utf-8"))
		# duration = ref_audio_len + int(ref_audio_len / ref_text_len * gen_text_len / speed)

	for j in range(1):
		print(f"Iniciando Inferencia {j}")
		# inference
		with torch.inference_mode():
			generated, _ = model_obj.sample(
				cond=audio,
				text=final_text_list,
				duration=duration,
				steps=nfe_step,
				# steps=64,
				cfg_strength=cfg_strength,
				sway_sampling_coef=sway_sampling_coef,
			)
			generated = generated.to(torch.float32)
			generated = generated[:, ref_audio_len:, :]

			# #########################################################33
			# if ref_audio_len >= generated.shape[1]:
			# 	print(f"[WARN] ref_audio_len ({ref_audio_len}) >= generated length ({generated.shape[1]}), skipping slice.")
			# 	generated_trimmed = generated  # o considera usar generated[:, -1:, :] como fallback
			# else:
			# 	generated_trimmed = generated[:, ref_audio_len:, :]
			# generated_mel_spec = generated_trimmed.permute(0, 2, 1)
			# # print("generated shape:", generated.shape)
			# # print("generated_mel_spec shape:", generated_mel_spec.shape)
			# ###########################################################

			generated_mel_spec = generated.permute(0, 2, 1)

			if mel_spec_type == "vocos":
				generated_wave = vocoder.decode(generated_mel_spec)
			elif mel_spec_type == "bigvgan":
				generated_wave = vocoder(generated_mel_spec)
			if rms < target_rms:
				generated_wave = generated_wave * rms / target_rms

			# wav -> numpy
			generated_wave = generated_wave.squeeze().cpu().numpy()

			generated_waves.append(generated_wave)
			spectrograms.append(generated_mel_spec[0].cpu().numpy())

			sf.write(f'{ruta}{i:03d}.{j}.wav', generated_wave,target_sample_rate)
	i+=1	


  0%|                                                                                                                                                                  | 0/87 [00:00<?, ?it/s]Building prefix dict from the default dictionary ...
Loading model from cache /tmp/jieba.cache
Loading model cost 0.493 seconds.
Prefix dict has been built successfully.


Iniciando Inferencia 0


  1%|█▊                                                                                                                                                        | 1/87 [00:04<07:02,  4.91s/it]

Iniciando Inferencia 0


  2%|███▌                                                                                                                                                      | 2/87 [00:09<06:28,  4.57s/it]

...

 98%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▍   | 85/87 [06:04<00:08,  4.37s/it]

Iniciando Inferencia 0


 99%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▏ | 86/87 [06:09<00:04,  4.58s/it]

Iniciando Inferencia 0


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 87/87 [06:13<00:00,  4.29s/it]


In [50]:
i=40
j=1
gen_text_batches=['Después de un tiempo, se levantó y caminó como si nada. Era él. Y no lo era.'.lower()]

# _ref_audio_cache = {}
# # load asr pipeline
# asr_pipe = None

for _, gen_text in enumerate(progress.tqdm(gen_text_batches)):
# Prepare the text
	print (gen_text)
	text_list = [ref_text + gen_text]
	final_text_list = convert_char_to_pinyin(text_list)

	ref_audio_len = audio.shape[-1] // hop_length
	if fix_duration is not None:
		duration = int(fix_duration * target_sample_rate / hop_length)
	else:
		# Calculate duration
		ref_text_len = len(ref_text.encode("utf-8"))
		gen_text_len = len(gen_text.encode("utf-8"))
		# print("ref_audio_len ",ref_audio_len)
		# print("ref_text_len ",ref_text_len)
		# print("gen_text_len ",gen_text_len)
		# print("speed ",speed)
		duration = ref_audio_len + int(ref_audio_len / ref_text_len * gen_text_len / speed)
		# print("duration ",duration)
		
		# ref_text_len = len(ref_text.encode("utf-8"))
		# gen_text_len = len(gen_text.encode("utf-8"))
		# duration = ref_audio_len + int(ref_audio_len / ref_text_len * gen_text_len / speed)

	for _ in range(3):
		print(f"Iniciando Inferencia {j}")
		# inference
		with torch.inference_mode():
			generated, _ = model_obj.sample(
				cond=audio,
				text=final_text_list,
				duration=duration,
				steps=nfe_step,
				cfg_strength=cfg_strength,
				sway_sampling_coef=sway_sampling_coef,
			)
			generated = generated.to(torch.float32)
			generated = generated[:, ref_audio_len:, :]

			#########################################################33
			if ref_audio_len >= generated.shape[1]:
				print(f"[WARN] ref_audio_len ({ref_audio_len}) >= generated length ({generated.shape[1]}), skipping slice.")
				generated_trimmed = generated  # o considera usar generated[:, -1:, :] como fallback
			else:
				generated_trimmed = generated[:, ref_audio_len:, :]

			generated_mel_spec = generated_trimmed.permute(0, 2, 1)

			###########################################################
			# print("generated shape:", generated.shape)
			# print("generated_mel_spec shape:", generated_mel_spec.shape)

			if mel_spec_type == "vocos":
				generated_wave = vocoder.decode(generated_mel_spec)
			elif mel_spec_type == "bigvgan":
				generated_wave = vocoder(generated_mel_spec)
			if rms < target_rms:
				generated_wave = generated_wave * rms / target_rms

			# wav -> numpy
			generated_wave = generated_wave.squeeze().cpu().numpy()

			generated_waves.append(generated_wave)
			spectrograms.append(generated_mel_spec[0].cpu().numpy())

			sf.write(f'{ruta}{i:02d}.{j}.wav', generated_wave,target_sample_rate)
		j+=1

  0%|                                                                                                                                                                   | 0/1 [00:00<?, ?it/s]

después de un tiempo, se levantó y caminó como si nada. era él. y no lo era.
Iniciando Inferencia 1
[WARN] ref_audio_len (1359) >= generated length (436), skipping slice.
Iniciando Inferencia 2
[WARN] ref_audio_len (1359) >= generated length (436), skipping slice.
Iniciando Inferencia 3


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:14<00:00, 14.89s/it]

[WARN] ref_audio_len (1359) >= generated length (436), skipping slice.


In [ ]:
# Combine all generated waves with cross-fading
if cross_fade_duration <= 0:
	# Simply concatenate
	final_wave = np.concatenate(generated_waves)
else:
	final_wave = generated_waves[0]
	for i in range(1, len(generated_waves)):
		prev_wave = final_wave
		next_wave = generated_waves[i]

		# Calculate cross-fade samples, ensuring it does not exceed wave lengths
		cross_fade_samples = int(cross_fade_duration * target_sample_rate)
		cross_fade_samples = min(cross_fade_samples, len(prev_wave), len(next_wave))

		if cross_fade_samples <= 0:
			# No overlap possible, concatenate
			final_wave = np.concatenate([prev_wave, next_wave])
			continue

		# Overlapping parts
		prev_overlap = prev_wave[-cross_fade_samples:]
		next_overlap = next_wave[:cross_fade_samples]

		# Fade out and fade in
		fade_out = np.linspace(1, 0, cross_fade_samples)
		fade_in = np.linspace(0, 1, cross_fade_samples)

		# Cross-faded overlap
		cross_faded_overlap = prev_overlap * fade_out + next_overlap * fade_in

		# Combine
		new_wave = np.concatenate(
			[prev_wave[:-cross_fade_samples], cross_faded_overlap, next_wave[cross_fade_samples:]]
		)

		final_wave = new_wave

# Create a combined spectrogram
combined_spectrogram = np.concatenate(spectrograms, axis=1)
final_sample_rate=target_sample_rate
# return final_wave, target_sample_rate, combined_spectrogram

In [ ]:
# if remove_silence:
with tempfile.NamedTemporaryFile(delete=False, suffix=".wav") as f:
	sf.write(f.name, final_wave, final_sample_rate)
	remove_silence_for_generated_wav(f.name)
	final_wave, _ = torchaudio.load(f.name)
final_wave = final_wave.squeeze().cpu().numpy()

In [ ]:
sf.write(ruta+'FuenteJuventudComplete.wav', final_wave,final_sample_rate)